# 🌱 CAC40 ESG Report Intelligence — RAG Pipeline

**Author:** Deepayan Sarkar  
**Date:** July 2026  
**Program:** MSc Data Analytics for Business — KEDGE Business School

---

## Project Overview

A Retrieval-Augmented Generation (RAG) pipeline that enables semantic search and CSRD compliance analysis across 14 CAC40 sustainability reports.

### Problem
With the EU's Corporate Sustainability Reporting Directive (CSRD) now mandatory, analysts must compare hundreds of pages of ESG reports across companies. Manual analysis is slow, inconsistent, and unscalable.

### Solution
An AI-powered tool that:
- Indexes 14 CAC40 ESG reports as vector embeddings in MongoDB Atlas
- Answers natural language questions across all reports simultaneously
- Automatically generates CSRD compliance gap analysis for any company

### Stack
| Component | Technology |
|---|---|
| Vector Database | MongoDB Atlas Vector Search |
| Embeddings | Voyage AI (voyage-3, 1024 dimensions) |
| LLM | Mistral AI (mistral-small-latest) |
| Interface | Streamlit (deployed live) |
| Language | Python 3.11 |

### Dataset
- **14 CAC40 companies** across 10 sectors
- **8,441 chunks** indexed with 1024-dimensional embeddings
- **12 CSRD requirements** assessed per company

### Live Demo
👉 https://cac40-esg-intelligence.streamlit.app

In [3]:
# Install necessary Python packages for MongoDB, Voyage AI, Mistral AI, and PDF processing.
!pip install pymongo voyageai mistralai pypdf2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-semantic-conventions==

In [5]:
import os
from mistralai.client import Mistral
import voyageai
from pymongo import MongoClient

# Paste your actual keys here. These are sensitive credentials.
MONGODB_URI = "MONGODB_URI"
VOYAGE_API_KEY = "VOYAGE_API_KEY"
MISTRAL_API_KEY = "MISTRAL_API_KEY"

# Test MongoDB connection to ensure it's working correctly.
client = MongoClient(MONGODB_URI)
print("MongoDB connected:", client.server_info()["version"])

# Test Voyage AI connection and get embedding size to confirm functionality.
vc = voyageai.Client(api_key=VOYAGE_API_KEY)
result = vc.embed(["test"], model="voyage-3")
print("Voyage AI connected: embedding size =", len(result.embeddings[0]))

# Test Mistral AI connection by sending a simple chat message.
mistral = Mistral(api_key=MISTRAL_API_KEY)
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[{"role": "user", "content": "Say hello in one word"}]
)
print("Mistral connected:", response.choices[0].message.content)

MongoDB connected: 8.0.28
Voyage AI connected: embedding size = 1024
Mistral connected: Hi


In [15]:
import PyPDF2

# Define a function to extract text from a PDF file.
def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        total_pages = len(reader.pages)
        print(f"Total pages: {total_pages}")
        for page_num, page in enumerate(reader.pages):
            text += page.extract_text() + "\n"
            if (page_num + 1) % 10 == 0:
                print(f"Processed {page_num + 1}/{total_pages} pages...")
    return text

In [ ]:
# Extract text from a sample PDF document for initial processing.
raw_text = extract_text_from_pdf("/content/totalenergies_sustainability-climate-2024-progress-report_2024_en_pdf.pdf")
print(f"\nTotal characters extracted: {len(raw_text)}")
print("\nFirst 500 characters preview:")
print(raw_text[:500])

Total pages: 112
Processed 10/112 pages...
Processed 20/112 pages...
Processed 30/112 pages...
Processed 40/112 pages...
Processed 50/112 pages...
Processed 60/112 pages...
Processed 70/112 pages...
Processed 80/112 pages...
Processed 90/112 pages...
Processed 100/112 pages...
Processed 110/112 pages...

Total characters extracted: 305700

First 500 characters preview:
More Energy,  
Less Emissions
Sustainability & Climate 2024 Progress Report
1  Sustainability & Climate 2024 Progress Report 1  Sustainability & Climate 2024 Progress Report
Our Approach to Sustainable Development
Energy is at the heart of the most daunting chal -
lenges of the 21st century, defined in the U.N.’s 
2030 Agenda in the form of its 17 Sustainable 
Development Goals (SDGs).
As part of its transition strategy to achieve its 
2050 Net Zero Ambition, together with society, 
the Company 


In [16]:
# Define a function to split raw text into smaller, overlapping chunks.
def split_into_chunks(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # overlap keeps context between chunks
    return chunks

In [ ]:
# Split the extracted raw text into manageable chunks.
chunks = split_into_chunks(raw_text)
print(f"Total chunks created: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[5])

Total chunks created: 383

Example chunk:
ty: an Integrated Approach 46
Our Renewable Electricity Capacity Build-up  47
Developing Electric Mobility  48
New Low-carbon Energy and Innovations  
to Achieve Net Zero by 2050  49
New Low-carbon Energy 49
Focus:  Sustainable Aviation Fuel (SAF) 51
Innovating to Accelerate the Energy Transition  52OUR ACTIONS  
FOR A JUST TRANSITION 54
Our Just Transition Plan  55
Advocacy and Sector Initiatives  
in Support of the Energy Transition 58
Acting for the Well-Being of Employees 60
Ensuring People’s Safety  61
Our Employees at the Heart of the Transition 64
Engaging Every Employee  65
Attracting, Developing and Retaining Talents 66
Building a Good Place to Work 67
Promoting Diversity and Inclusion 69
Sustainab’ALL  Program  70 
Stories.  Sustainab’ALL 71
Caring for the Environment 72
Caring for the Environment  73
Environmental Protection  74
Taking Action to Preserve Water Resources 75
Developing Circular Management of Our Products  77
Stories. S

In [18]:
import voyageai
from pymongo import MongoClient
import time

# Your API credentials for MongoDB and Voyage AI.
MONGODB_URI = "MONGODB_URI"
VOYAGE_API_KEY = "VOYAGE_API_KEY"

# Connect to MongoDB Atlas and select the database and collection.
client = MongoClient(MONGODB_URI)
db = client["esg_rag"]
collection = db["documents"]

In [ ]:
# Clear existing documents for 'TotalEnergies' to prevent duplicates if rerunning.
collection.delete_many({"company": "TotalEnergies"})
print("Cleared previous chunks")

# Connect to Voyage AI client for embedding generation.
vc = voyageai.Client(api_key=VOYAGE_API_KEY)

# Process and store text chunks with their embeddings into MongoDB.
company_name = "TotalEnergies"
batch_size = 10
total_stored = 0

print(f"Processing {len(chunks)} chunks...")

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    # Generate embeddings for the current batch of text chunks.
    result = vc.embed(batch, model="voyage-3")

    # Prepare documents for insertion into MongoDB.
    documents = []
    for j, (chunk, embedding) in enumerate(zip(batch, result.embeddings)):
        documents.append({
            "company": company_name,
            "chunk_index": i + j,
            "text": chunk,
            "embedding": embedding
        })

    # Insert the batch of documents into the MongoDB collection.
    collection.insert_many(documents)
    total_stored += len(batch)
    print(f"Stored {total_stored}/{len(chunks)} chunks...")

    # Pause to respect Voyage AI rate limits.
    time.sleep(20)

print(f"\nDone! {total_stored} chunks stored in MongoDB.")

Cleared previous chunks
Processing 383 chunks...
Stored 10/383 chunks...
Stored 20/383 chunks...
Stored 30/383 chunks...
Stored 40/383 chunks...
Stored 50/383 chunks...
Stored 60/383 chunks...
Stored 70/383 chunks...
Stored 80/383 chunks...
Stored 90/383 chunks...
Stored 100/383 chunks...
Stored 110/383 chunks...
Stored 120/383 chunks...
Stored 130/383 chunks...
Stored 140/383 chunks...
Stored 150/383 chunks...
Stored 160/383 chunks...
Stored 170/383 chunks...
Stored 180/383 chunks...
Stored 190/383 chunks...
Stored 200/383 chunks...
Stored 210/383 chunks...
Stored 220/383 chunks...
Stored 230/383 chunks...
Stored 240/383 chunks...
Stored 250/383 chunks...
Stored 260/383 chunks...
Stored 270/383 chunks...
Stored 280/383 chunks...
Stored 290/383 chunks...
Stored 300/383 chunks...
Stored 310/383 chunks...
Stored 320/383 chunks...
Stored 330/383 chunks...
Stored 340/383 chunks...
Stored 350/383 chunks...
Stored 360/383 chunks...
Stored 370/383 chunks...
Stored 380/383 chunks...
Stored 383

In [ ]:
# Import SearchIndexModel for creating vector search indexes in MongoDB.
from pymongo.operations import SearchIndexModel

# Define the vector search index model for efficient semantic search.
index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            },
            {
                "type": "filter",
                "path": "company"
            }
        ]
    },
    name="esg_embedding_index",
    type="vectorSearch"
)

# Create the vector search index in the MongoDB collection.
collection.create_search_index(model=index_model)
print("Vector search index created successfully")

Vector search index created successfully


In [10]:
# Define a function to perform an ESG search using vector similarity.
def search_esg(query, company=None, top_k=5):
    # Embed the user's query using Voyage AI.
    query_embedding = vc.embed([query], model="voyage-3").embeddings[0]

    # Construct the MongoDB aggregation pipeline for vector search.
    pipeline = [
        {
            "$vectorSearch": {
                "index": "esg_embedding_index",
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": top_k,
                "filter": {"company": company} if company else {}
            }
        },
        {
            "$project": {
                "company": 1,
                "text": 1,
                "score": {"$meta": "vectorSearchScore"},
                "_id": 0
            }
        }
    ]

    # Execute the aggregation pipeline and return the results.
    results = list(collection.aggregate(pipeline))
    return results

In [ ]:
# Test the ESG search function with a specific query for 'TotalEnergies'.
results = search_esg("What are TotalEnergies net zero targets?", company="TotalEnergies")

# Print the retrieved results, including the similarity score and a snippet of the text.
for i, result in enumerate(results):
    print(f"\n--- Result {i+1} (score: {result['score']:.4f}) ---")
    print(result['text'][:300])


--- Result 1 (score: 0.8011) ---
YSTEM ACCORDING TO THE IEA IN 2050
Total primary energy demand mix – Worlwide
TotalEnergies' sales mix
Electricity 
& Renewables Electricity 1
± 500 TWh/y
LNG & Gas 
OilCoal 
Natural gas
Oil
25-30 Mt/y LNG
0.2-0.3 Mb/d Oil10 Mt/y polymersLow-carbon 
molecules 2Bioenergy
± 50 Mt/y
11
10
23
30
4347822

--- Result 2 (score: 0.8009) ---
ergy represents the history 
and the future of TotalEnergies. Rising to the dual challenge 
of meeting the energy needs of an ever-growing world 
population while reducing global warming, reinventing 
energy production and consumption in order to get to Net 
Zero by 2050, together with society, thos

--- Result 3 (score: 0.7968) ---
companies had amended 
their practices to reflect this new concept1.HIGHLIGHTS  
70  Sustainability & Climate 2024 Progress Report 70  Sustainability & Climate 2024 Progress Report
TOTALENERGIES’ 
AMBITION IN SUPPORT 
OF SUSTAINABLE 
DEVELOPMENT
—
TotalEnergies’ ambition to be a major player 
in

In [ ]:
from mistralai.client import Mistral

MISTRAL_API_KEY = "MISTRAL_API_KEY"
mistral = Mistral(api_key=MISTRAL_API_KEY)

# Define a function to answer questions using a RAG approach.
def answer_question(query, company=None):
    # Step 1 — Retrieve relevant chunks from the vector database based on the query.
    results = search_esg(query, company=company)

    # Step 2 — Combine the retrieved text chunks to form a context for the LLM.
    context = "\n\n".join([r["text"] for r in results])

    # Step 3 — Construct the prompt for Mistral AI, instructing it to answer using only the provided context.
    prompt = f"""You are an ESG analyst assistant. Answer the question below using ONLY the context provided.
If the answer is not in the context, say "I could not find this information in the report."
Always cite specific numbers and targets when available.

Context:
{context}

Question: {query}

Answer:"""

    # Step 4 — Generate the answer using Mistral AI's chat completion model.
    response = mistral.chat.complete(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    # Print the question, the generated answer, and the number of sources used.
    print(f"Question: {query}")
    print(f"\nAnswer: {answer}")
    print(f"\nSources: {len(results)} chunks retrieved from {company or 'all companies'}")

# Test the answer_question function with a query for 'TotalEnergies'.
answer_question("What are TotalEnergies net zero targets?", company="TotalEnergies")

Question: What are TotalEnergies net zero targets?

Answer: TotalEnergies' net zero targets by 2050 include:

1. **Energy Production Mix**:
   - About **50% of its energy in the form of electricity**, including storage, totaling around **500 TWh/year** (developed from about **400 GW of gross renewable capacity**).
   - About **25% of its energy as low-carbon molecules** (e.g., biogas, hydrogen, or synthetic fuels), equivalent to **50 Mt/year**.
   - The remaining **25% would come from natural gas and oil**, with **oil reduced to 0.2–0.3 Mb/d** and **LNG at 25–30 Mt/year**.

2. **Emissions Reduction**:
   - **Scope 1 & 2 emissions** (from operated facilities) reduced to **zero (below 0.1 Mt CO₂e/year)**, offset by nature-based solutions.
   - **Scope 3 emissions** (from customers' use of products) targeted at **100 Mt CO₂e/year**, with efforts to "eliminate" an equivalent amount through **carbon utilization (CCU) and carbon capture and storage (CCS)** solutions.

3. **Corporate Commitme

In [19]:
import time
import PyPDF2

# Define a helper function to process a single company's PDF, extract text, chunk it, embed, and store in MongoDB.
def process_company(company_name, pdf_path):
    print(f"\n--- Processing {company_name} ---")

    # Step 1 — Extract text from the PDF.
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        print(f"Pages: {len(reader.pages)}")
        for page in reader.pages:
            text += page.extract_text() + "\n"
    print(f"Characters extracted: {len(text)}")

    # Step 2 — Split the extracted text into chunks.
    chunks = split_into_chunks(text)
    print(f"Chunks created: {len(chunks)}")

    # Step 3 — Embed chunks and store them in MongoDB in batches.
    batch_size = 10
    total_stored = 0

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        # Generate embeddings for the batch.
        result = vc.embed(batch, model="voyage-3")

        documents = []
        for j, (chunk, embedding) in enumerate(zip(batch, result.embeddings)):
            documents.append({
                "company": company_name,
                "chunk_index": i + j,
                "text": chunk,
                "embedding": embedding
            })

        collection.insert_many(documents)
        total_stored += len(batch)
        time.sleep(20) # Pause to respect Voyage AI rate limit.

    print(f"Stored {total_stored} chunks for {company_name}")
    return total_stored

In [ ]:
# Define a list of companies for the first batch of processing.
batch_1 = [
    {"name": "Airbus", "path": "/content/Airbus.pdf"},
    {"name": "AXA", "path": "/content/Axa.pdf"},
    {"name": "BNP Paribas", "path": "/content/bnp_paribas.pdf"},
    {"name": "Capgemini", "path": "/content/Capgemini.pdf"},
]

# Iterate through batch_1 and process each company's PDF.
for company in batch_1:
    try:
        process_company(company["name"], company["path"])
    except Exception as e:
        print(f"ERROR processing {company['name']}: {e}")
        continue

print("=== BATCH 1 COMPLETE ===")


--- Processing Airbus ---
Pages: 377
Characters extracted: 1467521
Chunks created: 1835
Stored 1835 chunks for Airbus

--- Processing AXA ---
Pages: 63
Characters extracted: 204788
Chunks created: 256
Stored 256 chunks for AXA

--- Processing BNP Paribas ---
Pages: 36
Characters extracted: 94311
Chunks created: 118
Stored 118 chunks for BNP Paribas

--- Processing Capgemini ---
Pages: 530
Characters extracted: 2229973
Chunks created: 2788


In [20]:
# Define a list of companies for the second batch of processing.
batch_2 = [
    {"name": "Danone", "path": "/content/Danone.pdf"},
    {"name": "Engie", "path": "/content/ENGIE.pdf"},
    {"name": "L'Oreal", "path": "/content/loreal.pdf"},
    {"name": "LVMH", "path": "/content/LVMH.pdf"},
]

# Iterate through batch_2 and process each company's PDF.
for company in batch_2:
    try:
        process_company(company["name"], company["path"])
    except Exception as e:
        print(f"ERROR processing {company['name']}: {e}")
        continue

print("=== BATCH 2 COMPLETE ===")


--- Processing Danone ---
Pages: 31
Characters extracted: 81131
Chunks created: 102
Stored 102 chunks for Danone

--- Processing Engie ---
Pages: 75
Characters extracted: 417284
Chunks created: 522
Stored 522 chunks for Engie

--- Processing L'Oreal ---
Pages: 144
Characters extracted: 144
Chunks created: 1
Stored 1 chunks for L'Oreal

--- Processing LVMH ---
Pages: 172
Characters extracted: 356717
Chunks created: 446
Stored 446 chunks for LVMH
=== BATCH 2 COMPLETE ===


In [21]:
# Define a list of companies for the third batch of processing.
batch_3 = [
    {"name": "Orange", "path": "/content/Orange.pdf"},
    {"name": "Renault", "path": "/content/Renault.pdf"},
    {"name": "Sanofi", "path": "/content/Sanofie.pdf"},
    {"name": "Schneider Electric", "path": "/content/Schneider.pdf"},
    {"name": "Stellantis", "path": "/content/Stellantis.pdf"},
]

# Iterate through batch_3 and process each company's PDF.
for company in batch_3:
    try:
        process_company(company["name"], company["path"])
    except Exception as e:
        print(f"ERROR processing {company['name']}: {e}")
        continue

print("=== BATCH 3 COMPLETE ===")


--- Processing Orange ---
Pages: 40
Characters extracted: 132502
Chunks created: 166
Stored 166 chunks for Orange

--- Processing Renault ---
Pages: 61
Characters extracted: 116412
Chunks created: 146
Stored 146 chunks for Renault

--- Processing Sanofi ---
Pages: 162
Characters extracted: 727923
Chunks created: 910
Stored 910 chunks for Sanofi

--- Processing Schneider Electric ---
Pages: 148
Characters extracted: 1279104
Chunks created: 1599
Stored 1599 chunks for Schneider Electric

--- Processing Stellantis ---
Pages: 131
Characters extracted: 464535
Chunks created: 581
Stored 581 chunks for Stellantis
=== BATCH 3 COMPLETE ===


In [30]:
# Delete a specific chunk for 'L'Oreal' in preparation for re-processing.
result = collection.delete_many({"company": "L'Oreal"})
print(f"Deleted {result.deleted_count} chunks for L'Oreal")

Deleted 1 chunks for L'Oreal


In [31]:
# Define a specific function to re-process L'Oreal's PDF with selected page ranges.
def process_loreal():
    print("\n--- Processing L'Oreal ---")

    text = ""
    with open("/content/loreal.pdf", "rb") as f:
        reader = PyPDF2.PdfReader(f)
        # Specify relevant page numbers to extract text from.
        relevant_pages = list(range(6, 52)) + list(range(186, 284))
        for page_num in relevant_pages:
            text += reader.pages[page_num].extract_text() + "\n"

    print(f"Characters extracted: {len(text)}")

    chunks = split_into_chunks(text)
    print(f"Chunks created: {len(chunks)}")

    # Delete any existing L'Oreal documents before re-inserting.
    collection.delete_many({"company": "L'Oreal"})

    batch_size = 10
    total_stored = 0

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        result = vc.embed(batch, model="voyage-3")

        documents = []
        for j, (chunk, embedding) in enumerate(zip(batch, result.embeddings)):
            documents.append({
                "company": "L'Oreal",
                "chunk_index": i + j,
                "text": chunk,
                "embedding": embedding
            })

        collection.insert_many(documents)
        total_stored += len(batch)
        print(f"Stored {total_stored}/{len(chunks)} chunks...")
        time.sleep(20) # Pause to respect Voyage AI rate limit.

    print(f"✅ Done! {total_stored} chunks stored for L'Oreal")

# Execute the L'Oreal specific processing function.
process_loreal()


--- Processing L'Oreal ---
Characters extracted: 517289
Chunks created: 647
Stored 10/647 chunks...
Stored 20/647 chunks...
Stored 30/647 chunks...
Stored 40/647 chunks...
Stored 50/647 chunks...
Stored 60/647 chunks...
Stored 70/647 chunks...
Stored 80/647 chunks...
Stored 90/647 chunks...
Stored 100/647 chunks...
Stored 110/647 chunks...
Stored 120/647 chunks...
Stored 130/647 chunks...
Stored 140/647 chunks...
Stored 150/647 chunks...
Stored 160/647 chunks...
Stored 170/647 chunks...
Stored 180/647 chunks...
Stored 190/647 chunks...
Stored 200/647 chunks...
Stored 210/647 chunks...
Stored 220/647 chunks...
Stored 230/647 chunks...
Stored 240/647 chunks...
Stored 250/647 chunks...
Stored 260/647 chunks...
Stored 270/647 chunks...
Stored 280/647 chunks...
Stored 290/647 chunks...
Stored 300/647 chunks...
Stored 310/647 chunks...
Stored 320/647 chunks...
Stored 330/647 chunks...
Stored 340/647 chunks...
Stored 350/647 chunks...
Stored 360/647 chunks...
Stored 370/647 chunks...
Stored 

In [32]:
# Aggregate and count the number of documents (chunks) stored for each company in MongoDB.
pipeline = [
    {"$group": {"_id": "$company", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]

results = list(collection.aggregate(pipeline))
for r in results:
    print(f"{r['_id']}: {r['count']} chunks")

AXA: 256 chunks
Airbus: 1835 chunks
BNP Paribas: 118 chunks
Capgemini: 730 chunks
Danone: 102 chunks
Engie: 522 chunks
L'Oreal: 647 chunks
LVMH: 446 chunks
Orange: 166 chunks
Renault: 146 chunks
Sanofi: 910 chunks
Schneider Electric: 1599 chunks
Stellantis: 581 chunks
TotalEnergies: 383 chunks


In [37]:
import time

# Define a function to perform CSRD gap analysis for a given company.
def analyze_company_csrd(company_name):
    print(f"\n{'='*50}")
    print(f"CSRD Gap Analysis: {company_name}")
    print(f"{'='*50}")

    results = []

    # Iterate through each CSRD requirement in the checklist.
    for requirement in csrd_checklist:
        # Retrieve top relevant chunks for the current requirement and company.
        chunks = search_esg(requirement, company=company_name, top_k=3)
        context = "\n\n".join([c["text"] for c in chunks])

        # Construct a prompt for Mistral AI to assess compliance based on the context.
        prompt = f"""You are a CSRD compliance expert analyzing ESG reports.

Based ONLY on the context below from {company_name}'s sustainability report, assess whether this CSRD requirement is met:

Requirement: {requirement}

Context:
{context}

Respond with EXACTLY one of these three options and a brief one-line reason:
✅ DISCLOSED - Clear evidence found
⚠️ PARTIAL - Some mention but incomplete
❌ MISSING - No evidence found

Format: [STATUS] - [reason]"""

        # Get the assessment from Mistral AI.
        response = mistral.chat.complete(
            model="mistral-small-latest",
            messages=[{"role": "user", "content": prompt}]
        )

        answer = response.choices[0].message.content.strip()
        print(f"{requirement[:50]}: {answer}")
        results.append({
            "requirement": requirement,
            "status": answer
        })

        time.sleep(25)  # Pause to respect Voyage AI rate limit.

    return results

In [38]:
# Run the CSRD gap analysis for 'TotalEnergies' using the defined function.
totalenergies_results = analyze_company_csrd("TotalEnergies")


CSRD Gap Analysis: TotalEnergies
Net zero or carbon neutrality target and timeline: ✅ DISCLOSED - Clear evidence of a net zero target by 2050 for direct emissions (Scope 1+2) and a strategy to address Scope 3 emissions through CCU/CCS and low-carbon energy solutions.
Scope 1 and Scope 2 greenhouse gas emissions data: ✅ DISCLOSED - Clear evidence found of Scope 1 and Scope 2 emissions data (e.g., "Scope 1+2 Mt CO2e 46 44 41*37*40 35") and breakdowns by geography/type.
Scope 3 value chain emissions disclosure: ✅ DISCLOSED - Clear evidence found of Scope 3 value chain emissions disclosure (e.g., Categories 1-6 estimates provided).
Renewable energy usage and targets: ✅ DISCLOSED - Clear evidence of renewable energy usage (22 GW installed capacity) and targets (35 GW by 2025, 100 GW by 2030) with specific projects (e.g., Cottonwood Bayou, Xlinks).
Water consumption and reduction targets: ✅ DISCLOSED - Clear evidence of a 20% reduction target for freshwater withdrawals at water-stressed sit

In [39]:
# Initialize a dictionary to store CSRD analysis results for all companies.
all_results = {}

# Define the list of companies to be analyzed.
companies = [
    "TotalEnergies", "Airbus", "AXA", "BNP Paribas",
    "Capgemini", "Danone", "Engie", "L'Oreal",
    "LVMH", "Orange", "Renault", "Sanofi",
    "Schneider Electric", "Stellantis"
]

# Iterate through each company and perform CSRD gap analysis.
for company in companies:
    try:
        all_results[company] = analyze_company_csrd(company)
    except Exception as e:
        print(f"ERROR processing {company}: {e}")
        continue

print("\n=== ALL COMPANIES ANALYZED ===")


CSRD Gap Analysis: TotalEnergies
Net zero or carbon neutrality target and timeline: ✅ DISCLOSED - Clear evidence of a net zero target by 2050 for Scope 1+2 emissions and a strategy to address Scope 3 emissions.
Scope 1 and Scope 2 greenhouse gas emissions data: ✅ DISCLOSED - Clear evidence found
Scope 3 value chain emissions disclosure: ⚠️ PARTIAL - Discloses some Scope 3 categories but lacks comprehensive value chain emissions coverage.
Renewable energy usage and targets: ✅ DISCLOSED - Clear evidence of renewable energy usage (22 GW installed capacity, 35 GW target by 2025, 100 GW by 2030) and specific projects (Cottonwood Bayou, Brazoria Solar, Xlinks) with quantified targets.
Water consumption and reduction targets: ✅ DISCLOSED - Clear evidence of a 20% reduction target for freshwater withdrawals at water-stressed sites by 2030, with progress updates and site-specific plans.
Biodiversity impact assessment and commitments: ✅ DISCLOSED - Clear evidence of biodiversity impact assessme

In [40]:
# Get the MongoDB collection for storing gap analysis results.
gap_collection = db["gap_analysis"]

# Store all the generated gap analysis results in MongoDB.
for company, results in all_results.items():
    # Delete any existing analysis for the company before inserting to ensure fresh data.
    gap_collection.delete_many({"company": company})
    gap_collection.insert_one({
        "company": company,
        "results": results,
        "generated_at": "2026-07-24" # Timestamp for when the analysis was generated.
    })
    print(f"Stored gap analysis for {company}")

print("\nAll gap analysis results stored in MongoDB")

Stored gap analysis for TotalEnergies
Stored gap analysis for Airbus
Stored gap analysis for AXA
Stored gap analysis for BNP Paribas
Stored gap analysis for Capgemini
Stored gap analysis for Danone
Stored gap analysis for Engie

All gap analysis results stored in MongoDB
